In [2]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os

# Take environment variables from .env
load_dotenv(override=True)

# This notebook uses the following variables from your .env file
search_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
knowledge_base_name = os.getenv("KNOWLEDGE_BASE_NAME")

In [3]:
# The Knowledge Base exposes an MCP endpoint for tool-based access
mcp_endpoint = f"{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version=2025-11-01-Preview"
print(f"✅ MCP endpoint: {mcp_endpoint}")
     

✅ MCP endpoint: https://rag-time.search.windows.net/knowledgebases/earth-knowledge-base/mcp?api-version=2025-11-01-Preview


In [8]:
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

credential = DefaultAzureCredential()

bearer_token_provider = get_bearer_token_provider(
    credential, "https://management.azure.com/.default"
)

headers = {
  "Authorization": f"Bearer {bearer_token_provider()}",
}

In [ ]:
import requests
from azure.identity import get_bearer_token_provider

FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME")
PROJECT_RESOURCE_ID = os.environ["FOUNDRY_PROJECT_RESOURCE_ID"]
PROJECT_CONNECTION_NAME = "earth-kb-mcp-connection"


conn_response = requests.put(
    f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview",
    headers=headers,
    json={
        "name": PROJECT_CONNECTION_NAME,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {"ApiType": "Azure"},
        },
    },
)
conn_response.raise_for_status()
print(f"✅ Project connection '{PROJECT_CONNECTION_NAME}' created")